# Deployment, monitoring, and governed agentic controls

Package evidence, calculate drift, and let an agent triage only within a deterministic permission boundary.

All data are generated locally unless this notebook explicitly calls a reviewed adapter. Results are educational and require independent validation before any real use.

In [ ]:
from creditriskbook.agents import GovernedMonitoringAgent
from creditriskbook.data.datasets import load_dataset
from creditriskbook.data.quality import assess_quality, inject_teaching_defects
from creditriskbook.monitoring import population_stability_index

bundle = load_dataset("synthetic_retail", n_rows=3_000, seed=707)
reference = bundle.frame.loc[bundle.frame["application_date"] < "2023-01-01", "utilisation"].to_numpy()
current = bundle.frame.loc[bundle.frame["application_date"] >= "2023-01-01", "utilisation"].to_numpy()
psi = population_stability_index(reference, current)
quality = assess_quality(bundle)
recommendation = GovernedMonitoringAgent().review(quality, {"pd_psi": psi, "roc_auc": 0.72})
print(recommendation.to_dict())
assert recommendation.human_approval_required
assert "approve_customer_credit" in recommendation.prohibited_actions

In [ ]:
dirty = inject_teaching_defects(bundle, seed=708)
halt = GovernedMonitoringAgent().review(assess_quality(bundle, dirty), {"pd_psi": 0.01, "roc_auc": 0.75})
assert halt.status == "HALT"
print(halt.recommended_action, halt.evidence_sha256)

An LLM may summarise evidence or draft a ticket. It may not invent evidence, change thresholds, approve credit, retrain, deploy, or close an incident without the separately authorised human workflow.